In [1]:
import pandas as pd

In [11]:
import sys

In [12]:
sys.path.append('..')

In [ ]:
df_questions = pd.read_csv('questions.csv')
df_questions.head()

,question,group,type
0,How do I set up LLM as a judge?,LLM evaluation,direct
1,I want to evaluate my chatbot responses,LLM evaluation,vague
2,How do I customize evaluation criteria for my ...,LLM evaluation,specific
3,What metrics are available?,Metrics and descriptors,broad
4,How do I measure text length?,Metrics and descriptors,specific descriptor


In [9]:
row = df_questions.iloc[0]
row.question

'How do I set up LLM as a judge?'

In [ ]:

from tools import create_documentation_tools_cached
from doc_agent import create_agent, DocumentationAgentConfig

tools = create_documentation_tools_cached()
agent_config = DocumentationAgentConfig()

agent = create_agent(agent_config, tools)

Loading documentation tools from .cache/documentation_tools.pkl...


In [17]:
from doc_agent import AgentStreamRunner
from jaxn import JSONParserHandler

async def run_agent(agent, user_prompt, message_history=None):
    runner = AgentStreamRunner(agent, JSONParserHandler())
    result = await runner.run(user_prompt, message_history)
    return result

In [18]:
result = await run_agent(agent, row.question)

USER PROMPT (search): How do I set up LLM as a judge?
TOOL CALL (search): search({"query":"LLM as a judge"})
TOOL CALL (search): get_file({"filename": "examples/LLM_judge.mdx"})
TOOL CALL (search): get_file({"filename": "examples/LLM_evals.mdx"})


In [ ]:

from tests.cost_tracker import calculate_cost


usage = result.usage()

cost = calculate_cost(
    model_name=agent_config.model,
    input_tokens=usage.input_tokens,
    output_tokens=usage.output_tokens
)

In [37]:
import dataclasses
from tests.utils import collect_tools

tools = collect_tools(result.all_messages())
tools_dicts = [dataclasses.asdict(t) for t in tools]

In [ ]:
rag_output = result.output

In [43]:
output = {
    'input': row.to_dict(),
    'rag_response': rag_output.model_dump(),
    'tools': tools_dicts,
    'cost': cost
}

In [45]:
async def run_eval_round(row):
    result = await run_agent(agent, row.question)

    usage = result.usage()

    cost = calculate_cost(
        model_name=agent_config.model,
        input_tokens=usage.input_tokens,
        output_tokens=usage.output_tokens
    )

    tools = collect_tools(result.all_messages())
    tools_dicts = [dataclasses.asdict(t) for t in tools]
    
    rag_output = result.output

    output = {
        'input': row.to_dict(),
        'rag_response': rag_output.model_dump(),
        'tools': tools_dicts,
        'cost': cost
    }

    return output

In [47]:
output = await run_eval_round(row)

USER PROMPT (search): How do I set up LLM as a judge?
TOOL CALL (search): search({"query":"LLM as a judge"})
TOOL CALL (search): get_file({"filename": "examples/LLM_judge.mdx"})
TOOL CALL (search): get_file({"filename": "examples/LLM_evals.mdx"})


In [48]:
output

{'input': {'question': 'How do I set up LLM as a judge?',
  'group': 'LLM evaluation',
  'type': 'direct'},
 'rag_response': {'answer': 'To set up an LLM as a judge, you will follow these steps to create an evaluator that can assess responses based on predefined criteria. Here\'s how to do it:\n\n### 1. Installation and Imports\nMake sure to install the Evidently package using:\n```python\nuv add evidently\n```\nThen, import the necessary modules:\n```python\nimport pandas as pd\nimport numpy as np\n\nfrom evidently import Dataset\nfrom evidently import DataDefinition\nfrom evidently import Report\nfrom evidently import BinaryClassification\nfrom evidently.descriptors import *\nfrom evidently.presets import TextEvals, ValueStats, ClassificationPreset\nfrom evidently.metrics import *\n\nfrom evidently.llm.templates import BinaryClassificationPromptTemplate\n```\n\n### 2. Pass Your OpenAI API Key\nSet your OpenAI API key as an environment variable:\n```python\nimport os\nos.environ["OPEN

In [55]:
from tqdm.auto import tqdm

In [59]:
import traceback

outputs = []

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    try:
        output = await run_eval_round(row)
        outputs.append(output)

    except Exception as e:
        print(f"Cannot process row {row}: {e}")
        traceback.print_exc()

  0%|          | 0/27 [00:00<?, ?it/s]

USER PROMPT (search): How do I set up LLM as a judge?
TOOL CALL (search): search({"query":"LLM as a judge"})
TOOL CALL (search): get_file({"filename": "examples/LLM_judge.mdx"})
TOOL CALL (search): get_file({"filename": "examples/LLM_evals.mdx"})
USER PROMPT (search): I want to evaluate my chatbot responses
TOOL CALL (search): search({"query":"evaluate chatbot responses"})
TOOL CALL (search): get_file({"filename": "examples/LLM_judge.mdx"})
TOOL CALL (search): get_file({"filename": "examples/LLM_evaluation.mdx"})
TOOL CALL (search): get_file({"filename":"quickstart_llm.mdx"})
USER PROMPT (search): How do I customize evaluation criteria for my use case?
TOOL CALL (search): search({"query":"customize evaluation criteria"})
TOOL CALL (search): get_file({"filename": "quickstart_llm.mdx"})
TOOL CALL (search): get_file({"filename": "metrics/customize_llm_judge.mdx"})
USER PROMPT (search): What metrics are available?
TOOL CALL (search): search({"query":"available metrics"})
TOOL CALL (search)

In [60]:
len(outputs)

27

In [61]:
import json

In [63]:
with open('evals_run_2026_03_06.json', 'wt') as f_out:
    json.dump(outputs, f_out, indent=2)

In [66]:
total_cost = sum([o['cost'] for o in outputs])
print(f'total cost: {total_cost:0.3f}')

total cost: 0.089
